# Normalization for Identity Resolution Phase: Wikidata

This notebook normalizes the Wikidata airport dataset (`wikidataMapped.xml`) for the identity resolution phase.

Steps applied:
1. Lowercase & remove punctuation
2. Strip spaces
3. Unicode normalization (replace ï, ö, ä, accented chars)
4. Translation *(SKIPPED – pending coaching session)*
5. Abbreviation expansion
6. Remove stopwords and common airport words
7. Country columns: add `countryCode` from `countryName` via pycountry fuzzy search

## 1. Lowercase & remove punctuation for string columns

Targeted columns: `name`, `countryName`, `city`

Removes: commas, dots, parentheses, `'`, `"`, hyphens (kept as space), and other punctuation.

> Note: Wikidata has no `countryCode` column — it will be derived in step 7.

## 2. Strip spaces

Remove leading/trailing whitespace and collapse multiple internal spaces.

## 3. Unicode normalization

Uses NFKD decomposition to separate base characters from diacritics, then encodes to ASCII (ignoring the diacritic combining characters).

Examples:
- ï, ö, ä → i, o, a
- São Paulo → sao paulo
- Piñon → pinon
- Çarşamba → carsamba

## 4. Translation *(SKIPPED FOR NOW: pending coaching session)*

Wikidata has many airports with non-English names. Discuss with professor whether translation is worthwhile.

## 5. Abbreviation expansion

The Wikidata dataset uses various abbreviations in the `name` column. The abbreviations below were identified by:
1. The known list from the task description
2. A frequency analysis of title-case and all-caps short tokens in the name column

Expansion is applied as **whole-word** substitution (word boundaries `\b`) to avoid accidental replacements inside longer words.

## 6. Remove stopwords and common words

After abbreviation expansion many names still contain common words like `international`, `airport`, `the`, `of`, etc. that add noise to string comparators.

Strategy: remove English function words and Romance/Germanic articles from the `name` column.

## 7. Country columns – add `countryCode` via pycountry fuzzy search

Wikidata has **only `countryName`** (no ISO code). Use pycountry fuzzy search to derive `countryCode`:

```python
import pycountry
results = pycountry.countries.search_fuzzy("People's Republic of China")
# → [Country(alpha_2='CN', name='China', ...)]
```

`search_fuzzy` returns a **list** — always take the first result (`[0]`). Wrap in try/except to handle names that cannot be matched.

This allows `countryCode` to be used as a blocking key across all three datasets.

## Final: Save normalized dataset